In [6]:
import torch
from datasets import load_dataset
import pandas as pd
import numpy as np

def inspectPathVqaDataset():
    print("⏳ Đang tải bộ dữ liệu PathVQA từ Hugging Face Hub (Vui lòng đợi)...")
    # Tải cả 3 tập để kiểm tra kích thước phân phối
    try:
        pathVqaTrain = load_dataset("flaviagiammarino/path-vqa", split="train")
        pathVqaVal = load_dataset("flaviagiammarino/path-vqa", split="validation")
        pathVqaTest = load_dataset("flaviagiammarino/path-vqa", split="test")
    except Exception as e:
        print(f"❌ Lỗi khi tải dữ liệu: {e}. Vui lòng kiểm tra lại kết nối Internet.")
        return

    print("\n" + "="*50)
    print("📊 BÁO CÁO THÔNG TIN BỘ DỮ LIỆU PATHVQA")
    print("="*50)
    print(f"  - Số lượng mẫu tập Train     : {len(pathVqaTrain)} câu hỏi")
    print(f"  - Số lượng mẫu tập Validation: {len(pathVqaVal)} câu hỏi")
    print(f"  - Số lượng mẫu tập Test       : {len(pathVqaTest)} câu hỏi")
    print(f"  - Tổng cộng                  : {len(pathVqaTrain)+len(pathVqaVal)+len(pathVqaTest)} cặp Q&A")
    print("-" * 50)

    # Kiểm tra cấu trúc phân phối dạng câu hỏi Closed (Yes/No) trong tập Test
    print("🔍 Đang phân tích ranh giới câu hỏi tập Test...")
    testAnswers = [str(item['answer']).strip().lower() for item in pathVqaTest]
    closedCount = sum(1 for ans in testAnswers if ans in ['yes', 'no'])
    openCount = len(pathVqaTest) - closedCount
    print(f"  - Số câu hỏi đóng (Yes/No)   : {closedCount} ({closedCount/len(pathVqaTest)*100:.2f}%)")
    print(f"  - Số câu hỏi mở (Generative) : {openCount} ({openCount/len(pathVqaTest)*100:.2f}%)")
    print("-" * 50)

    # Hiển thị ngẫu nhiên 3 mẫu dữ liệu để soi cấu trúc trường thông tin
    print("📋 Xem trước 3 mẫu ngẫu nhiên từ tập dữ liệu:")
    randomIndices = np.random.choice(len(pathVqaTrain), 3, replace=False)

    for rank, idx in enumerate(randomIndices):
        sample = pathVqaTrain[int(idx)]
        imgObj = sample['image']

        print(f"\n[{rank + 1}] Mẫu số ID: {idx}")
        print(f"    🔹 Câu hỏi: {sample['question']}")
        print(f"    🔸 Đáp án : {sample['answer']}")
        print(f"    🖼️ Chi tiết ảnh: Định dạng={imgObj.format} | Kích thước={imgObj.size} | Chế độ màu={imgObj.mode}")

if __name__ == "__main__":
    inspectPathVqaDataset()


⏳ Đang tải bộ dữ liệu PathVQA từ Hugging Face Hub (Vui lòng đợi)...

📊 BÁO CÁO THÔNG TIN BỘ DỮ LIỆU PATHVQA
  - Số lượng mẫu tập Train     : 19654 câu hỏi
  - Số lượng mẫu tập Validation: 6259 câu hỏi
  - Số lượng mẫu tập Test       : 6719 câu hỏi
  - Tổng cộng                  : 32632 cặp Q&A
--------------------------------------------------
🔍 Đang phân tích ranh giới câu hỏi tập Test...
  - Số câu hỏi đóng (Yes/No)   : 3362 (50.04%)
  - Số câu hỏi mở (Generative) : 3357 (49.96%)
--------------------------------------------------
📋 Xem trước 3 mẫu ngẫu nhiên từ tập dữ liệu:

[1] Mẫu số ID: 12456
    🔹 Câu hỏi: how is vascular changes and fibrosis of salivary glands produced?
    🔸 Đáp án : by radiation therapy of the neck region
    🖼️ Chi tiết ảnh: Định dạng=JPEG | Kích thước=(324, 432) | Chế độ màu=CMYK

[2] Mẫu số ID: 4148
    🔹 Câu hỏi: what is present?
    🔸 Đáp án : mycobacterium avium intracellulare
    🖼️ Chi tiết ảnh: Định dạng=JPEG | Kích thước=(792, 528) | Chế độ màu=RGB



In [7]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn datasets

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import open_clip
from datasets import load_dataset
from torch.utils.data import DataLoader
from PIL import ImageFile, Image
import os
from google.colab import drive

# Ép hệ thống bỏ qua lỗi ảnh thiếu byte của PathVQA
ImageFile.LOAD_TRUNCATED_IMAGES = True

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/PathVQA_RAG-2"
if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)

class GatedFusion(nn.Module):
    def __init__(self, embedDim=512):
        super(GatedFusion, self).__init__()
        self.gateLayer = nn.Linear(embedDim * 2, embedDim)
        self.sigmoidFunc = nn.Sigmoid()
        self.fusionLayer = nn.Linear(embedDim * 2, embedDim)

    def forward(self, imageEmbeds, textEmbeds):
        combinedEmbeds = torch.cat((imageEmbeds, textEmbeds), dim=1)
        gateValue = self.sigmoidFunc(self.gateLayer(combinedEmbeds))
        gatedImage = imageEmbeds * gateValue
        gatedText = textEmbeds * (1 - gateValue)
        fusedOutput = self.fusionLayer(torch.cat((gatedImage, gatedText), dim=1))
        fusedOutput = fusedOutput / fusedOutput.norm(dim=-1, keepdim=True)
        return fusedOutput

def trainPathVqaFusion():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Đang huấn luyện trên: {device}")

    trainDS = load_dataset("flaviagiammarino/path-vqa", split="train")

    modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    bioMedClip, _, preprocessImage = open_clip.create_model_and_transforms(modelName)
    tokenizer = open_clip.get_tokenizer(modelName)
    bioMedClip.to(device).eval()

    fusionModel = GatedFusion(embedDim=512).to(device)
    optimizer = optim.AdamW(fusionModel.parameters(), lr=2e-4, weight_decay=1e-5)
    lossFunction = nn.CrossEntropyLoss()

    # Định cấu hình tham số tối ưu
    numEpochs = 3 # Dữ liệu lớn nên 3 epochs là đủ để hội tụ ban đầu
    batchSize = 64
    temperature = 0.07

    def collateBatch(batchData):
        images_proc = []
        questions, answers = [], []
        for item in batchData:
            try:
                # Đảm bảo ảnh ở hệ màu RGB trước khi đưa vào BioMedCLIP
                img = item['image'].convert("RGB") if item['image'].mode != "RGB" else item['image']
                images_proc.append(preprocessImage(img))
                questions.append(str(item['question']))
                answers.append(str(item['answer']))
            except Exception:
                continue # Bỏ qua nếu có file ảnh hỏng nghiêm trọng
        return torch.stack(images_proc), tokenizer(questions), tokenizer(answers)

    dataLoader = DataLoader(trainDS, batch_size=batchSize, shuffle=True, collate_fn=collateBatch, drop_last=True)

    for epoch in range(numEpochs):
        fusionModel.train()
        totalLoss = 0.0
        for batchIdx, (images, questions, answers) in enumerate(dataLoader):
            images, questions, answers = images.to(device), questions.to(device), answers.to(device)
            optimizer.zero_grad()

            with torch.no_grad():
                imgFeats = bioMedClip.encode_image(images)
                questFeats = bioMedClip.encode_text(questions)
                ansFeats = bioMedClip.encode_text(answers)

                imgFeats /= imgFeats.norm(dim=-1, keepdim=True)
                questFeats /= questFeats.norm(dim=-1, keepdim=True)
                ansFeats /= ansFeats.norm(dim=-1, keepdim=True)

            fusionVec = fusionModel(imgFeats, questFeats)
            logits = (fusionVec @ ansFeats.T) / temperature
            labels = torch.arange(len(images)).to(device)

            loss = (lossFunction(logits, labels) + lossFunction(logits.T, labels)) / 2
            loss.backward()
            optimizer.step()

            totalLoss += loss.item()
            if batchIdx % 50 == 0:
                print(f"Epoch [{epoch+1}/{numEpochs}] - Batch {batchIdx}/{len(dataLoader)} - Loss: {loss.item():.4f}")

        print(f"--- Hoàn thành Epoch {epoch+1} | Loss Trung bình: {totalLoss/len(dataLoader):.4f} ---")

    torch.save(fusionModel.state_dict(), f"{SAVE_DIR}/pathvqa_gated_fusion.pth")
    print("✅ Đã huấn luyện và lưu thành công trọng số Gated Fusion cho PathVQA!")

if __name__ == "__main__":
    trainPathVqaFusion()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang huấn luyện trên: cuda
Epoch [1/3] - Batch 0/307 - Loss: 4.1782
Epoch [1/3] - Batch 50/307 - Loss: 3.2114
Epoch [1/3] - Batch 100/307 - Loss: 3.0098
Epoch [1/3] - Batch 150/307 - Loss: 3.1070
Epoch [1/3] - Batch 200/307 - Loss: 2.8840
Epoch [1/3] - Batch 250/307 - Loss: 2.8807
Epoch [1/3] - Batch 300/307 - Loss: 2.7451
--- Hoàn thành Epoch 1 | Loss Trung bình: 2.9931 ---
Epoch [2/3] - Batch 0/307 - Loss: 2.5391
Epoch [2/3] - Batch 50/307 - Loss: 2.6880
Epoch [2/3] - Batch 100/307 - Loss: 2.9324
Epoch [2/3] - Batch 150/307 - Loss: 2.7675
Epoch [2/3] - Batch 200/307 - Loss: 2.6660
Epoch [2/3] - Batch 250/307 - Loss: 2.5635
Epoch [2/3] - Batch 300/307 - Loss: 2.3451
--- Hoàn thành Epoch 2 | Loss Trung bình: 2.6934 ---
Epoch [3/3] - Batch 0/307 - Loss: 2.7225
Epoch [3/3] - Batch 50/307 - Loss: 2.4280
Epoch [3/3] - Batch 100/307 - Loss: 2.5816
Epoch [3/3] - Ba

In [9]:
import torch
import faiss
import open_clip
from datasets import load_dataset
import numpy as np
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
SAVE_DIR = "/content/drive/MyDrive/PathVQA_RAG"

def buildPathVqaVectorDatabase():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Xây dựng Database trên: {device}")

    modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    bioMedClip, _, preprocessImage = open_clip.create_model_and_transforms(modelName)
    tokenizer = open_clip.get_tokenizer(modelName)
    bioMedClip.to(device).eval()

    fusionModel = GatedFusion(embedDim=512).to(device)
    fusionModel.load_state_dict(torch.load(f"{SAVE_DIR}/pathvqa_gated_fusion.pth"))
    fusionModel.eval()

    trainDS = load_dataset("flaviagiammarino/path-vqa", split="train")
    faissIndex = faiss.IndexFlatL2(512)
    metaDataList = []

    print("Bắt đầu quy trình quét tạo lập Vector Database...")
    with torch.no_grad():
        for i, item in enumerate(trainDS):
            try:
                img = item['image'].convert("RGB") if item['image'].mode != "RGB" else item['image']
                procImg = preprocessImage(img).unsqueeze(0).to(device)
                tokens = tokenizer([str(item['question'])]).to(device)

                imgFeat = bioMedClip.encode_image(procImg)
                txtFeat = bioMedClip.encode_text(tokens)
                imgFeat /= imgFeat.norm(dim=-1, keepdim=True)
                txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

                fusionVec = fusionModel(imgFeat, txtFeat).cpu().numpy().astype('float32')
                faissIndex.add(fusionVec)

                metaDataList.append({"id": i, "question": item['question'], "answer": item['answer']})
            except Exception:
                continue # Đảm bảo tiến trình không bị ngắt quãng bởi ảnh hỏng

            if (i+1) % 2000 == 0:
                print(f"Đã lập chỉ mục: {i+1}/{len(trainDS)} mẫu...")

    faiss.write_index(faissIndex, f"{SAVE_DIR}/pathvqa_fusion.index")
    torch.save(metaDataList, f"{SAVE_DIR}/pathvqa_metadata.pt")
    print("🎉 HOÀN THÀNH GIAI ĐOẠN OFFLINE CHO PATHVQA!")

if __name__ == "__main__":
    buildPathVqaVectorDatabase()

Xây dựng Database trên: cuda
Bắt đầu quy trình quét tạo lập Vector Database...
Đã lập chỉ mục: 2000/19654 mẫu...
Đã lập chỉ mục: 4000/19654 mẫu...
Đã lập chỉ mục: 6000/19654 mẫu...
Đã lập chỉ mục: 8000/19654 mẫu...
Đã lập chỉ mục: 10000/19654 mẫu...
Đã lập chỉ mục: 12000/19654 mẫu...
Đã lập chỉ mục: 14000/19654 mẫu...
Đã lập chỉ mục: 16000/19654 mẫu...
Đã lập chỉ mục: 18000/19654 mẫu...
🎉 HOÀN THÀNH GIAI ĐOẠN OFFLINE CHO PATHVQA!
